# Food Atlas Data Quailty Assesment

## Accuracy
The dataset appears to be accurate. The FARA (Food Access Research Atlas) dataset was checked for negative values, and none appear in the dataset, which is an indicator of accurate data of this nature (counts, percentages and flags).

## Completeness
There are a significant amount of missing values (in this case, NaN values) in this dataset. The columns contain the bulk of the missing data, and there are no rows with all missing data. Some columns have over 41% missing values, such as LAPOP1_10 “Population count beyond 1 mile for urban areas or 10 miles for rural areas from supermarket”, which has 29957 missing values. To visualize this missing data, there are a few maps that show what counties are missing a specific column. Columns with a missing data overwhelming are columns that are using supermarket data, which makes sense. Presumably, the supermarket data may have been incomplete for more areas. 

## Timeliness
The Food Access Research Atlas is calculated based on census data from 2010 combined with other data from up to 2019. This data includes Urban or rural designation from 2019, data on income, vehicle availability, and SNAP participation from 2014-18 (American Community Survey), and lists of supermarkets, supercenters, and large grocery stores from 2019.
The population data is the most recent data available from the Food Access Research Atlas. Although food insecurity statistics are likely to have changed, as this data was before the 2020 COVID Pandemic, and it is now 2026, this data is the best available from an authoritative source. Also, counties, states, and FIPS codes will not change so all of those variables will still be accurate.  

Economic Research Service (ERS), U.S. Department of Agriculture (USDA). Food Access Research Atlas, https://www.ers.usda.gov/data-products/food-access-research-atlas/

## Consistency
The data appears to be consistent. The low-access population does not exceed total population, percentages are within [0,100], the values in the urban/rural flag columns were only 0 and 1, and there are no duplicate values in the CensusTract column. 


#### Load Data

In [1]:
import pandas as pd
import numpy as np

food_atlas = pd.read_csv("../data/Food Access Research Atlas.csv")

## Accuracy

### Negative Values check (immpossible data)

In [ ]:
print("ACCURACY")
numeric_cols = food_atlas.select_dtypes(include='number').columns
neg_report = {}

for col in numeric_cols:
    neg_mask = food_atlas[col] < 0
    cnt = neg_mask.sum()
    if cnt > 0:
        neg_report[col] = cnt

if neg_report:
    print("\n=== NEGATIVE VALUES FOUND ===")
    for col, cnt in neg_report.items():
        print(f"{col}: {cnt} negative values")
else:
    print("\nNo negative values found in any numeric column.")

ACCURACY

No negative values found in any numeric column.


## Completeness

### Missing Values check

In [ ]:
print("MISSING VALUE OVERVIEW")
missing = food_atlas.isnull().sum()
missing_pct = (food_atlas.isnull().mean() * 100).round(2)
missing_report = pd.DataFrame({
    'Missing_Count': missing,
    'Missing_Pct': missing_pct
})
# show cols with missing values
print(missing_report[missing_report['Missing_Count'] > 0])

#  identifying cols
crucial_cols = ['CensusTract', 'State', 'County', 'Pop2010']
for col in crucial_cols:
    if col in food_atlas.columns:
        if food_atlas[col].isnull().any():
            print(f"CRITICAL: {col} has missing values – {food_atlas[col].isnull().sum()} rows")
        else:
            print(f"{col}: complete")
    else:
        print(f"COLUMN {col} not found in dataset")

# empty rows
row_missing_all = food_atlas.isnull().all(axis=1).sum()
print(f"\nRows missing all values: {row_missing_all}")

MISSING VALUE OVERVIEW
                    Missing_Count  Missing_Pct
NUMGQTRS                       25         0.03
PCTGQTRS                       25         0.03
PovertyRate                     3         0.00
MedianFamilyIncome            748         1.03
LAPOP1_10                   29957        41.30
...                           ...          ...
TractAIAN                       4         0.01
TractOMultir                    4         0.01
TractHispanic                   4         0.01
TractHUNV                       4         0.01
TractSNAP                       4         0.01

[126 rows x 2 columns]
CensusTract: complete
State: complete
County: complete
Pop2010: complete

Rows missing all values: 0


### Zeros check

In [ ]:
print("ZERO VALUE OVERVIEW")
zero_count = (food_atlas == 0).sum()
zero_pct = ((food_atlas == 0).mean() * 100).round(2)
zero_report = pd.DataFrame({
    'Zero_Count': zero_count,
    'Zero_Pct': zero_pct
})
# show cols with zero values
print(zero_report[zero_report['Zero_Count'] > 0])

# identifying crucial cols
crucial_cols = ['CensusTract', 'State', 'County', 'Pop2010']
for col in crucial_cols:
    if col in food_atlas.columns:
        if (food_atlas[col] == 0).any():
            print(f"CRITICAL: {col} has zero values – {(food_atlas[col] == 0).sum()} rows")
        else:
            print(f"{col}: no zeros")
    else:
        print(f"COLUMN {col} not found in dataset")

# rows where every value is 0
row_all_zero = (food_atlas == 0).all(axis=1).sum()
print(f"\nRows containing all zeros: {row_all_zero}")

ZERO VALUE OVERVIEW
                   Zero_Count  Zero_Pct
Urban                   17362     23.94
OHU2010                   106      0.15
GroupQuartersFlag       72015     99.29
NUMGQTRS                27235     37.55
PCTGQTRS                27235     37.55
...                       ...       ...
TractAIAN                1495      2.06
TractOMultir              140      0.19
TractHispanic             129      0.18
TractHUNV                2452      3.38
TractSNAP                2040      2.81

[142 rows x 2 columns]
CensusTract: no zeros
State: no zeros
County: no zeros
Pop2010: no zeros

Rows containing all zeros: 0


In [ ]:
col_desc = {
    "CensusTract": "Census tract",
    "State": "State name",
    "County": "County name",
    "Urban": "Flag for urban tract",
    "POP2010": "Population count from 2010 census",
    "OHU2010": "Occupied housing unit count from 2010 census",
    "GroupQuartersFlag": "Flag for tract where >=67%",
    "NUMGQTRS": "Count of tract population residing in group quarters",
    "PCTGQTRS": "Percent of tract population residing in group quarters",
    "LILATracts_1And10": "Flag for low-income and low access when considering low accessibilty at 1 and 10 miles",
    "LILATracts_halfAnd10": "Flag for low-income and low access when considering low accessibilty at 1/2 and 10 miles",
    "LILATracts_1And20": "Flag for low-income and low access when considering low accessibilty at 1 and 20 miles",
    "LILATracts_Vehicle": "Flag for low-income and low access when considering vehicle access or at 20 miles",
    "HUNVFlag": "Flag for tract where >= 100 of households do not have a vehicle, and beyond 1/2 mile from supermarket",
    "LowIncomeTracts": "Flag for low income tract",
    "PovertyRate": "Share of the tract population living with income at or below the Federal poverty thresholds for family size",
    "MedianFamilyIncome": "Tract median family income",
    "LA1and10": "Flag for low access tract at 1 mile for urban areas or 10 miles for rural areas",
    "LAhalfand10": "Flag for low access tract at 1/2 mile for urban areas or 10 miles for rural areas",
    "LA1and20": "Flag for low access tract at 1 mile for urban areas or 20 miles for rural areas",
    "LATracts_half": "Flag for low access tract when considering 1/2 mile distance",
    "LATracts1": "Flag for low access tract when considering 1 mile distance",
    "LATracts10": "Flag for low access tract when considering 10 mile distance",
    "LATracts20": "Flag for low access tract when considering 20 mile distance",
    "LATractsVehicle_20": "Flag for tract where >= 100 of households do not have a vehicle, and beyond 1/2 mile from supermarket; or >= 500 individuals are beyond 20 miles from supermarket ; or >= 33% of individuals are beyond 20 miles from supermarket",
    "LAPOP1_10": "Population count beyond 1 mile for urban areas or 10 miles for rural areas from supermarket",
    "LAPOP05_10": "Population count beyond 1/2 mile for urban areas or 10 miles for rural areas from supermarket",
    "LAPOP1_20": "Population count beyond 1 mile for urban areas or 20 miles for rural areas from supermarket",
    "LALOWI1_10": "Low income population count beyond 1 mile for urban areas or 10 miles for rural areas from supermarket",
    "LALOWI05_10": "Low income population count beyond 1/2 mile for urban areas or 10 miles for rural areas from supermarket",
    "LALOWI1_20": "Low income population count beyond 1 mile for urban areas or 20 miles for rural areas from supermarket",
    "lapophalf": "Population count beyond 1/2 mile from supermarket",
    "lapophalfshare": "Share of tract population that are beyond 1/2 mile from supermarket",
    "lalowihalf": "Low income population count beyond 1/2 mile from supermarket",
    "lalowihalfshare": "Share of tract population that are low income individuals beyond 1/2 mile from supermarket",
    "lakidshalf": "Kids population count beyond 1/2 mile from supermarket",
    "lakidshalfshare": "Share of tract population that are kids beyond 1/2 mile from supermarket",
    "laseniorshalf": "Seniors population count beyond 1/2 mile from supermarket",
    "laseniorshalfshare": "Share of tract population that are seniors beyond 1/2 mile from supermarket",
    "lawhitehalf": "White population count beyond 1/2 mile from supermarket",
    "lawhitehalfshare": "Share of tract population that are white beyond 1/2 mile from supermarket",
    "lablackhalf": "Black or African American population count beyond 1/2 mile from supermarket",
    "lablackhalfshare": "Share of tract population that are Black or African American beyond 1/2 mile from supermarket",
    "laasianhalf": "Asian population count beyond 1/2 mile from supermarket",
    "laasianhalfshare": "Share of tract population that are Asian beyond 1/2 mile from supermarket",
    "lanhopihalf": "Native Hawaiian or Other Pacific Islander population count beyond 1/2 mile from supermarket",
    "lanhopihalfshare": "Share of tract population that are Native Hawaiian or Other Pacific Islander beyond 1/2 mile from supermarket",
    "laaianhalf": "American Indian or Alaska Native population count beyond 1/2 mile from supermarket",
    "laaianhalfshare": "Share of tract population that are American Indian or Alaska Native beyond 1/2 mile from supermarket",
    "laomultirhalf": "Other/Multiple race population count beyond 1/2 mile from supermarket",
    "laomultirhalfshare": "Share of tract population that are Other/Multiple race beyond 1/2 mile from supermarket",
    "lahisphalf": "Hispanic or Latino ethnicity population count beyond 1/2 mile from supermarket",
    "lahisphalfshare": "Share of tract population that are of Hispanic or Latino ethnicity beyond 1/2 mile from supermarket",
    "lahunvhalf": "Housing units without vehicle count beyond 1/2 mile from supermarket",
    "lahunvhalfshare": "Share of tract housing units that are without vehicle and beyond 1/2 mile from supermarket",
    "lasnaphalf": "Housing units receiving SNAP benefits count beyond 1/2 mile from supermarket",
    "lasnaphalfshare": "Share of tract housing units receiving SNAP benefits count beyond 1/2 mile from supermarket",
    "lapop1": "Population count beyond 1 mile from supermarket",
    "lapop1share": "Share of tract population that are beyond 1 mile from supermarket",
    "lalowi1": "Low income population count beyond 1 mile from supermarket",
    "lalowi1share": "Share of tract population that are low income individuals beyond 1 mile from supermarket",
    "lakids1": "Kids population count beyond 1 mile from supermarket",
    "lakids1share": "Share of tract population that are kids beyond 1 mile from supermarket",
    "laseniors1": "Seniors population count beyond 1 mile from supermarket",
    "laseniors1share": "Share of tract population that are seniors beyond 1 mile from supermarket",
    "lawhite1": "White population count beyond 1 mile from supermarket",
    "lawhite1share": "Share of tract population that are white beyond 1 mile from supermarket",
    "lablack1": "Black or African American population count beyond 1 mile from supermarket",
    "lablack1share": "Share of tract population that are Black or African American beyond 1 mile from supermarket",
    "laasian1": "Asian population count beyond 1 mile from supermarket",
    "laasian1share": "Share of tract population that are Asian beyond 1 mile from supermarket",
    "lanhopi1": "Native Hawaiian or Other Pacific Islander population count beyond 1 mile from supermarket",
    "lanhopi1share": "Share of tract population that are Native Hawaiian or Other Pacific Islander beyond 1 mile from supermarket",
    "laaian1": "American Indian or Alaska Native population count beyond 1 mile from supermarket",
    "laaian1share": "Share of tract population that are American Indian or Alaska Native beyond 1 mile from supermarket",
    "laomultir1": "Other/Multiple race population count beyond 1 mile from supermarket",
    "laomultir1share": "Share of tract population that are Other/Multiple race beyond 1 mile from supermarket",
    "lahisp1": "Hispanic or Latino ethnicity population count beyond 1 mile from supermarket",
    "lahisp1share": "Share of tract population that are of Hispanic or Latino ethnicity beyond 1 mile from supermarket",
    "lahunv1": "Housing units without vehicle count beyond 1 mile from supermarket",
    "lahunv1share": "Share of tract housing units that are without vehicle and beyond 1 mile from supermarket",
    "lasnap1": "Housing units receiving SNAP benefits count beyond 1 mile from supermarket",
    "lasnap1share": "Share of tract housing units receiving SNAP benefits count beyond 1 mile from supermarket",
    "lapop10": "Population count beyond 10 miles from supermarket",
    "lapop10share": "Share of tract population that are beyond 10 miles from supermarket",
    "lalowi10": "Low income population count beyond 10 miles from supermarket",
    "lalowi10share": "Share of tract population that are low income individuals beyond 10 miles from supermarket",
    "lakids10": "Kids population count beyond 10 miles from supermarket",
    "lakids10share": "Share of tract population that are kids beyond 10 miles from supermarket",
    "laseniors10": "Seniors population count beyond 10 miles from supermarket",
    "laseniors10share": "Share of tract population that are seniors beyond 10 miles from supermarket",
    "lawhite10": "White population count beyond 10 miles from supermarket",
    "lawhite10share": "Share of tract population that are white beyond 10 miles from supermarket",
    "lablack10": "Black or African American population count beyond 10 miles from supermarket",
    "lablack10share": "Share of tract population that are Black or African American beyond 10 miles from supermarket",
    "laasian10": "Asian population count beyond 10 miles from supermarket",
    "laasian10share": "Share of tract population that are Asian beyond 10 miles from supermarket",
    "lanhopi10": "Native Hawaiian or Other Pacific Islander population count beyond 10 miles from supermarket",
    "lanhopi10share": "Share of tract population that are Native Hawaiian or Other Pacific Islander beyond 10 miles from supermarket",
    "laaian10": "American Indian or Alaska Native population count beyond 10 miles from supermarket",
    "laaian10share": "Share of tract population that are American Indian or Alaska Native beyond 10 miles from supermarket",
    "laomultir10": "Other/Multiple race population count beyond 10 miles from supermarket",
    "laomultir10share": "Share of tract population that are Other/Multiple race beyond 10 miles from supermarket",
    "lahisp10": "Hispanic or Latino ethnicity population count beyond 10 miles from supermarket",
    "lahisp10share": "Share of tract population that are of Hispanic or Latino ethnicity beyond 10 miles from supermarket",
    "lahunv10": "Housing units without vehicle count beyond 10 miles from supermarket",
    "lahunv10share": "Share of tract housing units that are without vehicle and beyond 10 miles from supermarket",
    "lasnap10": "Housing units receiving SNAP benefits count beyond 10 miles from supermarket",
    "lasnap10share": "Share of tract housing units receiving SNAP benefits count beyond 10 miles from supermarket",
    "lapop20": "Population count beyond 20 miles from supermarket",
    "lapop20share": "Share of tract population that are beyond 20 miles from supermarket",
    "lalowi20": "Low income population count beyond 20 miles from supermarket",
    "lalowi20share": "Share of tract population that are low income individuals beyond 20 miles from supermarket",
    "lakids20": "Kids population count beyond 20 miles from supermarket",
    "lakids20share": "Share of tract population that are kids beyond 20 miles from supermarket",
    "laseniors20": "Seniors population count beyond 20 miles from supermarket",
    "laseniors20share": "Share of tract population that are seniors beyond 20 miles from supermarket",
    "lawhite20": "White population count beyond 20 miles from supermarket",
    "lawhite20share": "Share of tract population that are white beyond 20 miles from supermarket",
    "lablack20": "Black or African American population count beyond 20 miles from supermarket",
    "lablack20share": "Share of tract population that are Black or African American beyond 20 miles from supermarket",
    "laasian20": "Asian population count beyond 20 miles from supermarket",
    "laasian20share": "Share of tract population that are Asian beyond 20 miles from supermarket",
    "lanhopi20": "Native Hawaiian or Other Pacific Islander population count beyond 20 miles from supermarket",
    "lanhopi20share": "Share of tract population that are Native Hawaiian or Other Pacific Islander beyond 20 miles from supermarket",
    "laaian20": "American Indian or Alaska Native population count beyond 20 miles from supermarket",
    "laaian20share": "Share of tract population that are American Indian or Alaska Native beyond 20 miles from supermarket",
    "laomultir20": "Other/Multiple race population count beyond 20 miles from supermarket",
    "laomultir20share": "Share of tract population that are Other/Multiple race beyond 20 miles from supermarket",
    "lahisp20": "Hispanic or Latino ethnicity population count beyond 20 miles from supermarket",
    "lahisp20share": "Share of tract population that are of Hispanic or Latino ethnicity beyond 20 miles from supermarket",
    "lahunv20": "Housing units without vehicle count beyond 20 miles from supermarket",
    "lahunv20share": "Share of tract housing units that are without vehicle and beyond 20 miles from supermarket",
    "lasnap20": "Housing units receiving SNAP benefits count beyond 20 miles from supermarket",
    "lasnap20share": "Share of tract housing units receiving SNAP benefits count beyond 20 miles from supermarket",
    "TractLOWI": "Total count of low-income population in tract",
    "TractKids": "Total count of children age 0-17 in tract",
    "TractSeniors": "Total count of seniors age 65+ in tract",
    "TractWhite": "Total count of White population in tract",
    "TractBlack": "Total count of Black or African American population in tract",
    "TractAsian": "Total count of Asian population in tract",
    "TractNHOPI": "Total count of Native Hawaiian and Other Pacific Islander population in tract",
    "TractAIAN": "Total count of American Indian and Alaska Native population in tract",
    "TractOMultir": "Total count of Other/Multiple race population in tract",
    "TractHispanic": "Total count of Hispanic or Latino population in tract",
    "TractHUNV": "Total count of housing units without a vehicle in tract",
    "TractSNAP": "Total count of housing units receiving SNAP benefits in tract"
}

### Supermarket Missing Data

In [ ]:
supermarket_flag = []
missing_pct_vals = []

for col in food_atlas.columns:
    desc = col_desc.get(col, "")               
    is_sm = "supermarket" in desc.lower() if pd.notna(desc) else False
    supermarket_flag.append(is_sm)
    
    pct = food_atlas[col].isnull().mean() * 100
    missing_pct_vals.append(pct)

group_df = pd.DataFrame({
    'column': food_atlas.columns,
    'missing_pct': missing_pct_vals,
    'in_supermarket_desc': supermarket_flag
})

sm_pct = group_df.loc[group_df['in_supermarket_desc'], 'missing_pct']
other_pct = group_df.loc[~group_df['in_supermarket_desc'], 'missing_pct']

print("\nSupermarket Missing Value Percentages")
print(f"Columns 'supermarket' data: {len(sm_pct)}")
print(f"  Mean missing %: {sm_pct.mean():.2f}  (std {sm_pct.std():.2f})")
print(f"Columns without 'supermarket' data: {len(other_pct)}")
print(f"  Mean missing %: {other_pct.mean():.2f}  (std {other_pct.std():.2f})")


Supermarket Missing Value Percentages
Columns 'supermarket' data: 112
  Mean missing %: 53.29  (std 38.94)
Columns without 'supermarket' data: 36
  Mean missing %: 0.03  (std 0.17)


## Consistency

In [ ]:
print("CONSISTENCY")

# logic: low-access population must not exceed total population
pop_low_access_pairs = [
    ('Pop2010', 'LAPOP1_10'),
    ('Pop2010', 'TractLAPOP1'),
    ('Pop2010', 'TractLAPOP10'),
]
for pop_col, la_col in pop_low_access_pairs:
    if pop_col in food_atlas.columns and la_col in food_atlas.columns:
        mask = (food_atlas[la_col] > food_atlas[pop_col]) & food_atlas[pop_col].notna() & food_atlas[la_col].notna()
        violations = mask.sum()
        if violations > 0:
            print(f"CONSISTENCY: {la_col} > {pop_col} for {violations} tracts")
            print(food_atlas.loc[mask, [pop_col, la_col]].head())

# consistency: share fields should be within [0,100] (percentages) and possibly sum to 1 for related groups
share_cols = ['LAhalfshare', 'LA1and10share']
for col in share_cols:
    if col in food_atlas.columns:
        out_of_bounds = ((food_atlas[col] < 0) | (food_atlas[col] > 100)) & food_atlas[col].notna()
        print(f"{col}: {out_of_bounds.sum()} values outside [0,100]")


# consistency: urban/rural flag (Urban column) – should be 0 or 1
if 'Urban' in food_atlas.columns:
    valid = food_atlas['Urban'].isin([0,1])
    print(f"Urban flag invalid: {(~valid).sum()} rows")

# consistency: no duplicates on CensusTract
if 'CensusTract' in food_atlas.columns:
    dup_count = food_atlas['CensusTract'].duplicated().sum()
    print(f"\nDuplicate CensusTract entries: {dup_count}")
    if dup_count > 0:
        print(food_atlas[food_atlas['CensusTract'].duplicated(keep=False)].head())

CONSISTENCY
Urban flag invalid: 0 rows

Duplicate CensusTract entries: 0
